<div>
    <img src="AER-logo.PNG" width="500" class="center">
</div>

# PyFluent Tutorial on Converging Nozzle

This tutorial helps you setup an Ansys Fluent<sup>TM</sup>, fluid simulation software case for a Pitot tube in PyAnsys. This is not a stand alone tutorial. This is an alternative way of setting up the problem. You can find the Ansys Fluent GUI setup as part of the Teaching Package (https://innovationspace.ansys.com/product/simple-approximations-of-fluid-flows/). In order to understand and run the following tutorial, you need to have prerequisites:

1. Understanding of the Pythonic interface.
2. Ansys Fluent R24.2. (if you have older versions of Ansys Fluent software please update the version as arguments while using lauch_fluent command.)
3. Python 3.11.4

If you want help in installing the packages necessary please follow the instructions at the page: https://docs.pyansys.com/version/dev/getting_started.html

Understanding how to run Ansys Flunet simulations in Python would be an essential skill that will make future simulations easier to setup and run. This tool will help setup multiple simulations at a single time which will greatly help in setting up time which is valuable for students and Professors equally.  

The Following pythonic code can be run using the following IDEs (Integrated Developement environment):
1. Visual Studio Code
2. Jupyter Notebook
3. Spyder

## Physics corresponding to the Tutorial

### Bernoulli's Principle

The conservation of energy to frictionless laminar flow leads to a relationship between pressure and velocity. This equation is famously known as the Bernoulli's Principle. Basically it is a very restrictive simplified version of the conservation of energy equation with assumptions:
1. Steady State Flow
2. Incompressible irrotational flow
3. No heat transfered to or from the fluid.
4. No work is done on or by the fluid.

These assumptions together when applied to the Navier Stokes Equations ends up being the Bernoulli's Principle given by 
$$
    P + \rho \frac{U^2}{2} + B = constant
$$
where 
- $P$ = Internal Energy
- $\rho (U^2/2)$  = Kinetic Energy
- $B$ = Potential Energy

If we are neglecting the Body forces in this equation, we can end up with an equation for the total pressure of the body which is 
$$ 
    p_{tot} = p + \rho \frac{U^2}{2}
$$

where again
- $p$ = static Pressure
- $\rho (U^2/2)$ = is the Kinetic energy term with the pressure dimension which is also called the dynamic pressure.


## Pitot Tube
The Pitot tube is an essential sensor for determining the Airplane speed. Once the airplane is actually flying, the velocity of the flight can ba calculated by the help of measuring the total and static pressure and also the temperature of the external flowing air. This is where the Beroulli's equation comes in hand. A pitot tube is a simple device that has a hollow tube in which air stagnates which leads to total pressure values. it also contains a static port which measures the static pressure. These values together with the density of the air obtained from temperature measurement could be used in bernoulli's Principle to obtain the effective velocity of the aircraft using the equation that was defined the previous secction.

The following sections will give a detailed explanation on how this problem can be setup using the PyFluent API.


## Importing PyFluent module and relevant modules

This section uses the Ansys libraries to import relevant tools which will be utilised in this simulation example. The Pyfluent module and the visulaization modules.


In [ ]:
import ansys.fluent.core as pyfluent
from ansys.fluent.visualization import Graphics, Plots, set_config

## Launching Ansys Fluent software for simulation.

The Launch command have more arguments that could be given. More detailed explanation for the launch command could be obtained by following the link: https://fluent.docs.pyansys.com/version/stable/api/launcher/launcher.html#ansys.fluent.core.launcher.launcher.launch_fluent

If you do not want to open the GUI of the Ansys Fluent Window, please dont include the "ui_mode = "gui" in the arguments. We define the entire Ansys fluent setup into class called solver. This makes the future setup of simulation easier as all the properties and subclass within the Ansys fluent GUI can be accessed directly using the variable "solver".

However, In this simulation, it is best to open the GUI in the background as the some results can only be obtained from the console of the Ansys Simulatio tool GUI.

In [ ]:
solver = pyfluent.launch_fluent(product_version="24.2", dimension=3, precision="double", processor_count=4, ui_mode="gui")

## Reading the Mesh and checking the Mesh

The following commands can be used to read the file given along with this tutorial and display the mesh in the interface.

Note: Graphics api is used to display the mesh for the same.

In [ ]:
solver.file.read_mesh(file_name="pitot_tube.msh")
solver.mesh.check()

#Displaying the Mesh in the Pythonic Interface.
set_config(blocking=True, set_view_on_display="xy")
graphics = Graphics(session=solver)   #using the graphics command to create a graphics module.  This will be used in future to obtain results as well.

#Creating a module for mesh display in the window.
mesh1 = graphics.Meshes["mesh-1"]
mesh1.show_edges = True
mesh1.surfaces_list = ['z-coordinate-7', 'pitot-tube-external-wall', 'total-pressure-port', 'static-pressure-port', 'pitot-tube-internal-wall']
mesh1.display("window-1")



## Simulation Setup

### General Setup


The next step is to define the general simulation conditions. A pressure-based steady state simulation is carried out for the example and hence we define the appropriate variables accordingly for the setup.

In [ ]:
solver.setup.general.solver.type = "pressure-based"
solver.setup.general.solver.velocity_formulation= "absolute"
solver.setup.general.solver.time= "steady"
solver.setup.general.operating_conditions.operating_pressure = 0


In [ ]:
solver.setup.models.viscous.model='laminar'

### Material properties and Cell zone setup

In this section, we check if the materials used is air with the properties:
1. Density $= 0.8636 kg/m^3$
2. viscosity $=  1.678e-05$

Since the values are different we change thses values for the material properties of air.

##### Note:

get_state() is a command which could be used to obtain the current/ default vales correspondiong to the module. In this case materials. This command can be used if you are not sure what the current values are for a particular module and update the module accordingly.

In [ ]:
air = solver.setup.materials.fluid["air"]
air.density.option = 'constant'
air.density.value = 0.8636
air.viscosity.value = 1.678e-05
air.get_state()

In [ ]:
solver.setup.materials.get_state()

## Boundary Conditions

### Enclosure Sides

We give a specified shear wall condition for the enclosure walls with value 0 to mimic maximum slip conditions.


### Inlet

The inlet boundary conditions is velocity inlet and have:

- velocity = $62 m/s$
- Initial Guage pressure = $65780 Pa$

### Outlet
The outlet guage pressure should also be same as the inlet guage pressure which is equa; to = $65780 Pa$


Note - Make sure the rest of the boundary conditions are correct like the pilot walls are marked as walls with No slip etc.

To check you can use the statement print_state() which will return the current state of all the boiundary conditions.

In [ ]:
enclosure = solver. setup.boundary_conditions.wall["enclosure-sides"]
enclosure.momentum.shear_condition = 'Specified Shear'


inlet = solver.setup.boundary_conditions.velocity_inlet["inlet"]
inlet.momentum.velocity.value = 62
inlet.momentum.initial_gauge_pressure.value = 65780


outlet = solver.setup.boundary_conditions.pressure_outlet["outlet"]
outlet.momentum.gauge_pressure.value = 65780

solver.setup.boundary_conditions.print_state()


## Solution Methods and Solution Controls

The next we need to check if all the solution methods are correctly defined. It is to be noted that the warped face gradient correction is automatically enabled because of the presence of polyhedral cells in the mesh. this is necessary for the best representation and accurate calculations in such situations to reduce mesh errors.

The Solution controls are kept at default.


In [ ]:
solver.solution.methods.get_state()

### Residual Convergence criteria

For this simulation a convergence criteria of $1e^{-05}$ is used for all the variables. The convergence criteria should be dependant on a simulation basis. Some residuals might take longer time to reach the limit and leads to longer simulation time. Hence the residual convergece criteria needs to be defined accordingly. As a hand rule a good criteria for convergence is $1e^{-05}$. Once the convergence criteria is set, proceeed to run the itieration by giving an iteration number. Iteration is limit is given so that the simulation ends either the iteration count is reached or the convergence limit is reached.

In [ ]:
solver.solution.monitor.residual.equations["continuity"].absolute_criteria= (1e-05)
solver.solution.monitor.residual.equations["x-velocity"].absolute_criteria= (1e-05)
solver.solution.monitor.residual.equations["y-velocity"].absolute_criteria= (1e-05)
solver.solution.monitor.residual.equations["z-velocity"].absolute_criteria= (1e-05)

### Report Definitions

In [ ]:
solver.solution.report_definitions.surface["inlet-pressure"] = {}
solver.solution.report_definitions.surface["inlet-pressure"].report_type = "surface-areaavg"
solver.solution.report_definitions.surface["inlet-pressure"].field="pressure"
solver.solution.report_definitions.surface["inlet-pressure"].surface_names= "inlet"
solver.solution.report_definitions.surface["inlet-pressure"].per_surface=False

solver.solution.monitor.report_files["inlet-pressure"] = {
    "file_name": r".\\inlet-pressure.out",
    "report_defs": ["inlet-pressure"],
}

solver.solution.report_definitions.surface["velocity_outlet"] = {}
solver.solution.report_definitions.surface["velocity_outlet"].report_type = "surface-areaavg"
solver.solution.report_definitions.surface["velocity_outlet"].field="velocity-magnitude"
solver.solution.report_definitions.surface["velocity_outlet"].surface_names= "outlet"
solver.solution.report_definitions.surface["velocity_outlet"].per_surface=False

solver.solution.monitor.report_files["velocity_outlet"] = {
    "file_name": r".\\velocity_outlet.out",
    "report_defs": ["velocity_outlet"],
}

### Initialization

We are using Hybrid Initialization to initialize the simulation., However, in order to obtain convergence faster in this case, we also enable "Use of External Aero Favourable Settings". This is a very common method to implement this for external aerodynamics methods in order to accelerate the convergence of the scalar equation-0. More on this can be read at https://ansyshelp.ansys.com/public/account/secured?returnurl=/Views/Secured/corp/v242/en/flu_ug/flu_ug_sec_init_hybrid.html.


More over, FMG (Full MultiGraid) initialization is carrieed out to further get better initliazation values for the domain leading to faster convergence rate. This is very common practice in external aerodynamics solution to reeuce the overall cost of simulation. However, it is to be noted that this initialization is a bit time consuming but reduces the overall cost of simulations.

The FMG initialization takes the initalised simualtion and runs the simulation in coarser grid and interpolates the results into the finer grid that. 

In [ ]:
solver.solution.initialization.hybrid_init_options.general_settings.external_aero ="True"



solver.solution.initialization.initialize()
solver.solution.initialization.fmg.fmg_initialize()


#### Pseudo transient Settings

The Length scale is first set for the external diameter of the pitot tube and then to the internal tubes of pitot tubes., This gives a better chance of better convergence as well anda very common practice for external aerodynamics problem.

In [ ]:

solver.solution.run_calculation.pseudo_time_settings.time_step_method.length_scale_methods = "user-specified"

length_scale = [0.01,0.002]
for length in length_scale:
    solver.solution.run_calculation.pseudo_time_settings.time_step_method.length_scale = length
    solver.solution.run_calculation.iterate(iter_count=150)

## Results And Discussions

### Residual Plots

We first plot the residual plots. This is necessary to understand if the solution has converged to a satisfactory degree of accuracy. It can be seen that the once the length scale was reduced , all the residuals have maintained the same values. This shows that a psedo convergence is obtained which suggests that the simulation is completed.

In [ ]:
matplotlib_plot = Plots(solver)
residual = matplotlib_plot.Monitors["residual"]
residual.monitor_set_name = "residual"
residual.plot("window-1")

### Velocity and Pressure Value contours

The velocity and pressure plots are created on the ceterline plane through the domain given asn 'z-coordinate-7'. 

- The velocity Contour - the contour shows that there is zero velocity inside the inetrnal pipes of the Pitot tube. In the centerline, it cna be seen that the air cannot move forward and hence comes to a stop and also that the air flows parallel to the static pressure ports and does not also enter through the other two pipes.

- The Pressure Contour - It is seen that the due to the velocity of the air inside the central line pipe, the pressure values increases till it reaches the total pressure of the fluid. It can also be seen that the two pipes other than the center pipe has lower pressure and it is equal to the static pressure of the fluid.

In [ ]:
velocity_contour = graphics.Contours["contour-velocity"]
velocity_contour.field = "velocity-magnitude"
velocity_contour.surfaces_list = ["z-coordinate-7"]
velocity_contour.display("window-2")

velocity_contour = graphics.Contours["contour-velocity"]
velocity_contour.field = "pressure"
velocity_contour.surfaces_list = ["z-coordinate-7"]
velocity_contour.display("window-3")

### Obtaining the Velocity

In order to obtain the velocity of the object, we obtain the statiuc and total pressure values at the different ports. these values can be obtained by reading them from the console of the Ansys Fluent GUI as of now. The following command can be used to obtain the value on the console.

In [ ]:
solver.results.report.surface_integrals.area_weighted_avg(report_of = "pressure",surface_names = ["static-pressure-port", "total-pressure-port"])


### Further calculations

In order to obtain Velocity valus, Please use Bernoulli's Principle and check if the value obtained is corresposnding to the effective velocity that was given as boundary Conditions

#### Results 

Pressure  = $65783.214 Pa$

Total Pressure = $67389.955 Pa$

Usiong Bernoulli's Principle, the equation of Velocity is 


$$
    U = \sqrt{\frac{2(P_{tot} - p_s)}{\rho}} = \sqrt{\frac{2(67462.567 - 65771.241)}{0.8636}}

    U = 61.00025 m/s
$$

This shows that the value obtained is approx 1.5% off the effective value that was defined in the Boundary Conditions.

### Saving the case and Data files and exiting the simulation.

In [ ]:
solver.file.write(file_type = "case-data", file_name = "PitotTube")

solver.exit()

## Conclusion

Pitot tube is simple device that is used to obtain the velocity of an aircraft or other moving object by utilizing the Bernoulli's pricinple. The above simulationmimics a Pitot Tube simulation and by the help of the simulation, we were able to visulaize the pressure and velocity contour inside the central line of the tube and from there was able to obtain a velocity value that is accurate to a degfree of 1.5% from the defined velocity. The detailed explanation of the Physics and the Bernoulli'sm equation can be read through by following link. (https://innovationspace.ansys.com/product/simple-approximations-of-fluid-flows/). 


This example could be considered as a way of understanding and learning how to setup a case using PyAnsys. For more examples and documentation you can refer to the follwoing link (https://fluent.docs.pyansys.com/version/stable/index.html)
